# Find the customer types

*Data Analysis in Python · Topic 11.4*

Your CRM team suspects the store has customer types that RFM can't see: customers who only buy with a coupon, and customers who only buy gifts. Describe every customer with five behavior columns, cluster them with k-means, and profile the clusters.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell and the cell that reads the `valid_orders` table: the orders of 2024 and 2025 that weren't cancelled, with two more columns. `coupon_used` is `True` for an order with a coupon, and `gift_season` for an order placed from October 15 to December 24.
2. In the first cell marked ✏️, create the `features` table: one row per customer, with the customer IDs as the index and these columns, in this order:
   - `orders`: the customer's orders
   - `aov`: the mean of the customer's `subtotal`
   - `coupon_share`: the share of the customer's orders with a coupon
   - `gift_share`: the share of the customer's orders in the gift season
   - `tenure`: the whole days from the customer's first order to their last
3. In the second cell marked ✏️, create `scaled`: the `features` table with `orders` and `aov` in logs and `tenure` as `np.log1p()`, because it can be 0, scaled with `StandardScaler`.
4. In the third cell marked ✏️, create two dictionaries with the numbers of clusters from 2 to 8 as their keys: `inertias`, each k-means model's inertia, and `silhouettes`, its silhouette score. Fit every model with `n_init=10` and `random_state=0`, and score a sample with `sample_size=10000` and `random_state=0`. Then create `best_k`: the number of clusters with the highest silhouette score.
5. In the last cell marked ✏️, create `model`: k-means with `best_k` clusters, fit the same way. Add its labels to the `features` table as `cluster`. Then create the `profile` table: one row per cluster, with `customers`, the number of customers, and the mean of each of the five columns.
6. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `features` table has 41,776 customers and 5 columns, and `scaled` is a 41,776 × 5 array. The silhouette score is highest at 4 clusters, about `0.5219`, so `best_k` is `4`. The `profile` table has 4 clusters of 18,915, 10,193, 9,995 and 2,673 customers, and in the smallest one, 99.9% of the orders used a coupon.

## Parameters

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
CUTOFF = "2026-01-01"  # The timestamps are in UTC: the orders before it are the orders of 2024 and 2025.

In [ ]:
# The orders of 2024 and 2025 that weren't cancelled, with coupon_used and gift_season. Run it, and leave it as it is.
orders = pd.read_csv(DATA_URL + "orders.csv", parse_dates=["created_at"],
                     usecols=["order_id", "customer_id", "status", "subtotal", "coupon", "created_at"])
orders = orders[orders["created_at"] < CUTOFF]
valid_orders = orders[orders["status"] != "Cancelled"].copy()
valid_orders["coupon_used"] = valid_orders["coupon"].notna()
month_day = valid_orders["created_at"].dt.month * 100 + valid_orders["created_at"].dt.day
valid_orders["gift_season"] = month_day.between(1015, 1224)
valid_orders.shape

## Five columns per customer

In [ ]:
features = valid_orders.groupby("customer_id").agg(orders=("order_id", "count"), aov=("subtotal", "mean"),
                                                   coupon_share=("coupon_used", "mean"), gift_share=("gift_season", "mean"),
                                                   first_order=("created_at", "min"), last_order=("created_at", "max"))
features["tenure"] = (features["last_order"] - features["first_order"]).dt.days
features = features[["orders", "aov", "coupon_share", "gift_share", "tenure"]]
features.describe().round(2)

## Logs and scaling

In [ ]:
logged = features.assign(orders=np.log(features["orders"]), aov=np.log(features["aov"]), tenure=np.log1p(features["tenure"]))
scaled = StandardScaler().fit_transform(logged)
scaled.shape

## How many clusters?

In [ ]:
inertias = {}
silhouettes = {}
for k in range(2, 9):
    candidate = KMeans(n_clusters=k, n_init=10, random_state=0).fit(scaled)
    inertias[k] = candidate.inertia_
    silhouettes[k] = silhouette_score(scaled, candidate.labels_, sample_size=10000, random_state=0)
best_k = max(silhouettes, key=silhouettes.get)
best_k, silhouettes

## The clusters

In [ ]:
model = KMeans(n_clusters=best_k, n_init=10, random_state=0).fit(scaled)
features["cluster"] = model.labels_
profile = features.groupby("cluster").agg(customers=("orders", "size"), orders=("orders", "mean"), aov=("aov", "mean"),
                                           coupon_share=("coupon_share", "mean"), gift_share=("gift_share", "mean"),
                                           tenure=("tenure", "mean"))
profile.round(3)

In [ ]:
# The customer types the store was built with. Read them once your check passes.
segments = pd.read_csv(DATA_URL + "reveals/customer_segments.csv", index_col="customer_id")
revealed = features.merge(segments, left_index=True, right_index=True, how="left", validate="one_to_one")
pd.crosstab(revealed["cluster"], revealed["segment"], margins=True)

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import numpy as np
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    columns = ["orders", "aov", "coupon_share", "gift_share", "tenure"]
    sums = {"orders": 61702, "aov": 3535513.386025, "coupon_share": 3281.438095, "gift_share": 14215.081488, "tenure": 2481025}
    features = found.get("features")
    features_ok = False
    if not isinstance(features, pd.DataFrame) or [column for column in columns if column not in features.columns]:
        results.append((False, f"features should be a DataFrame with the columns {', '.join(columns)}."))
    elif len(features) != 41776:
        results.append((False, f"The features table should have one row per customer in the valid_orders table: 41,776. It has {len(features):,}."))
    else:
        wrong = [column for column in columns if abs(features[column].sum() - sums[column]) > 0.01]
        if wrong:
            column = wrong[0]
            hints = {"orders": "the count of the customer's orders", "aov": "the mean of the customer's subtotal",
                     "coupon_share": "the mean of coupon_used", "gift_share": "the mean of gift_season",
                     "tenure": "(last order - first order).dt.days"}
            results.append((False, f"features' {column} should be {hints[column]}: it adds up to {sums[column]:,.2f} over all customers. It adds up to {features[column].sum():,.2f}."))
        else:
            features_ok = True
            results.append((True, "41,776 customers: 7.9% of their orders used a coupon on average, and 34.0% were in the gift season."))

    scaled = found.get("scaled")
    if features_ok:
        logged = features[columns].assign(orders=np.log(features["orders"]), aov=np.log(features["aov"]), tenure=np.log1p(features["tenure"]))
        expected = ((logged - logged.mean()) / logged.std(ddof=0)).to_numpy()
        raw = features[columns].to_numpy()
        no_logs = ((raw - raw.mean(axis=0)) / raw.std(axis=0))
    if not isinstance(scaled, (np.ndarray, pd.DataFrame)) or np.shape(scaled) != (41776, 5):
        results.append((False, "scaled should be a 41,776 × 5 array: StandardScaler().fit_transform() of the five columns, with the logs."))
    elif features_ok and np.allclose(np.asarray(scaled, dtype=float), raw):
        results.append((False, "scaled is the features table as it is. Scale it with StandardScaler().fit_transform(), so every column has a mean of 0 and a standard deviation of 1."))
    elif features_ok and np.allclose(np.asarray(scaled, dtype=float), no_logs, atol=0.000001):
        results.append((False, "scaled has no logs: the few customers with many orders or a high order value pull the clusters. Take np.log() of orders and aov, and np.log1p() of tenure, before you scale."))
    elif features_ok and not np.allclose(np.asarray(scaled, dtype=float), expected, atol=0.000001):
        results.append((False, "scaled should be the five columns, with np.log() of orders and aov and np.log1p() of tenure, in the brief's order, scaled with StandardScaler."))
    elif features_ok:
        results.append((True, "Five columns on one scale: each has a mean of 0 and a standard deviation of 1."))

    inertias, silhouettes, best_k = found.get("inertias"), found.get("silhouettes"), found.get("best_k")
    expected_inertias = {2: 136270.246, 3: 99222.2313, 4: 65141.7456, 5: 52294.9081, 6: 44602.9626, 7: 37122.6033, 8: 32795.9124}
    expected_silhouettes = {2: 0.422696, 3: 0.48189, 4: 0.521938, 5: 0.504507, 6: 0.490176, 7: 0.497225, 8: 0.504701}
    if not isinstance(inertias, dict) or not isinstance(silhouettes, dict) or sorted(inertias) != list(range(2, 9)) or sorted(silhouettes) != list(range(2, 9)):
        results.append((False, "inertias and silhouettes should be dictionaries with the keys 2 to 8: {k: … for k in range(2, 9)}."))
    elif any(abs(inertias[k] - expected_inertias[k]) > 0.01 for k in expected_inertias):
        k = next(k for k in expected_inertias if abs(inertias[k] - expected_inertias[k]) > 0.01)
        results.append((False, f"inertias[{k}] should be about {expected_inertias[k]:,.2f}: KMeans(n_clusters={k}, n_init=10, random_state=0).fit(scaled).inertia_. It is {inertias[k]!r}."))
    elif any(abs(silhouettes[k] - expected_silhouettes[k]) > 0.00001 for k in expected_silhouettes):
        k = next(k for k in expected_silhouettes if abs(silhouettes[k] - expected_silhouettes[k]) > 0.00001)
        results.append((False, f"silhouettes[{k}] should be about {expected_silhouettes[k]:.4f}: silhouette_score(scaled, labels, sample_size=10000, random_state=0). It is {silhouettes[k]!r}."))
    elif best_k == 8:
        results.append((False, "best_k is 8, the lowest inertia, which always falls as you add clusters. Choose the highest silhouette score: max(silhouettes, key=silhouettes.get)."))
    elif best_k != 4:
        results.append((False, f"best_k should be the key of the highest silhouette score, 4. It is {best_k!r}."))
    else:
        results.append((True, "The silhouette score peaks at 4 clusters, 0.522, and the inertia's fall slows there too."))

    profile = found.get("profile")
    if not isinstance(features, pd.DataFrame) or "cluster" not in features.columns:
        results.append((False, "The features table should have a cluster column: model.labels_."))
    elif not isinstance(profile, pd.DataFrame) or "customers" not in profile.columns or [column for column in columns if column not in profile.columns]:
        results.append((False, "profile should be a DataFrame with one row per cluster, the column customers and the mean of each of the five columns."))
    elif sorted(int(n) for n in profile["customers"]) != [2673, 9995, 10193, 18915]:
        results.append((False, f"profile's customers should be 2,673, 9,995, 10,193 and 18,915. They are {sorted(int(n) for n in profile['customers'])}."))
    elif abs(profile.loc[profile["customers"] == 2673, "coupon_share"].iloc[0] - 0.999439) > 0.0001:
        results.append((False, "profile should have the mean of each column per cluster: in the cluster of 2,673 customers, 99.9% of the orders used a coupon."))
    else:
        results.append((True, "Four types: 2,673 customers who buy with a coupon, 9,995 who bought once in the gift season, 18,915 who bought once outside it, and 10,193 who came back, 2.9 orders on average."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run. Open the solution to see the customer types the store was built with.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
features = valid_orders.groupby("customer_id").agg(orders=("order_id", "count"), aov=("subtotal", "mean"),
                                                   coupon_share=("coupon_used", "mean"), gift_share=("gift_season", "mean"),
                                                   first_order=("created_at", "min"), last_order=("created_at", "max"))
features["tenure"] = (features["last_order"] - features["first_order"]).dt.days
features = features[["orders", "aov", "coupon_share", "gift_share", "tenure"]]
features.describe().round(2)
```

```python
logged = features.assign(orders=np.log(features["orders"]), aov=np.log(features["aov"]), tenure=np.log1p(features["tenure"]))
scaled = StandardScaler().fit_transform(logged)
scaled.shape
```

```python
inertias = {}
silhouettes = {}
for k in range(2, 9):
    candidate = KMeans(n_clusters=k, n_init=10, random_state=0).fit(scaled)
    inertias[k] = candidate.inertia_
    silhouettes[k] = silhouette_score(scaled, candidate.labels_, sample_size=10000, random_state=0)
best_k = max(silhouettes, key=silhouettes.get)
best_k, silhouettes
```

```python
model = KMeans(n_clusters=best_k, n_init=10, random_state=0).fit(scaled)
features["cluster"] = model.labels_
profile = features.groupby("cluster").agg(customers=("orders", "size"), orders=("orders", "mean"), aov=("aov", "mean"),
                                           coupon_share=("coupon_share", "mean"), gift_share=("gift_share", "mean"),
                                           tenure=("tenure", "mean"))
profile.round(3)
```

```python
# The customer types the store was built with. Read them once your check passes.
segments = pd.read_csv(DATA_URL + "reveals/customer_segments.csv", index_col="customer_id")
revealed = features.merge(segments, left_index=True, right_index=True, how="left", validate="one_to_one")
pd.crosstab(revealed["cluster"], revealed["segment"], margins=True)
```

```markdown
**The customer types the store was built with.** The data behind this course was generated with four hidden customer types: loyal high-value customers, discount hunters, gift buyers and lapsed customers. `reveals/customer_segments.csv` has each customer's type, and the crosstab above lines them up with your clusters.

The coupon cluster is pure: its 2,673 customers are all discount hunters. The repeat cluster holds 3,059 of the 3,853 loyal customers, 79%. But most customers ordered once, and a gift buyer's one order looks the same as a lapsed customer's, so both one-order clusters mix the two, split by the season of the order rather than by type. Neither k-means nor any other method can tell them apart from orders alone, until they order again or don't. A cluster describes the behavior you can see. It isn't the customer's type.
```

</details>